# Cleanup — Manage Graph Instances

Run this notebook to inspect and delete active graph instances / wrapper pods.

**Options:**
- Delete **all** active instances (Section 2)
- Delete instances older than **N hours** (Section 3 — bulk delete with filters)
- Delete by **owner**, **name prefix**, or any combination

Use `dry_run=True` to preview what would be deleted before committing.

In [ ]:
# ── Step 1 │ SDK Init ──────────────────────────────────────────────
# Services : graph_olap_sdk (local)  ·  Control Plane (:8080)
# ──────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, "/home/jovyan/work")

from graph_olap_sdk import GraphOLAPClient

# Admin client — can see and delete all instances
client = GraphOLAPClient(
    api_url="http://graph-olap-control-plane:8080",
    username="demo@example.com",
    role="admin",
)

print(client)

## 1 — List All Active Instances

In [ ]:
# ── Step 2 │ List Active Instances ─────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
import datetime

all_instances = client.instances.list()
active_statuses = {"running", "starting", "waiting_for_snapshot"}
active = [i for i in all_instances if i["status"] in active_statuses]

if not active:
    print("No active instances found — nothing to clean up.")
else:
    now = datetime.datetime.now(datetime.timezone.utc)
    print(f"Found {len(active)} active instance(s):\n")
    print(f"  {'ID':<8}  {'Status':<25}  {'Owner':<25}  {'Age':>8}  Pod")
    print(f"  {'-'*8}  {'-'*25}  {'-'*25}  {'-'*8}  {'-'*30}")
    for inst in active:
        age_str = "-"
        if inst.get("created_at"):
            created = datetime.datetime.fromisoformat(inst["created_at"].replace("Z", "+00:00"))
            age_h   = (now - created).total_seconds() / 3600
            age_str = f"{age_h:.1f}h"
        print(f"  {inst['id']:<8}  {inst['status']:<25}  {inst.get('created_by','-'):<25}  {age_str:>8}  {inst.get('pod_name','-')}")

## 2 — Delete ALL Active Instances

In [ ]:
# ── Step 3 │ Delete All Active Instances ───────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
import time

if not active:
    print("Nothing to delete.")
else:
    for inst in active:
        client.instances.terminate(inst["id"])

    time.sleep(8)

    remaining = [i for i in client.instances.list() if i["status"] in active_statuses]
    if remaining:
        print(f"Still active: {[i['id'] for i in remaining]}")
    else:
        print("All instances cleaned up — no wrapper pods running.")

## 3 — Bulk Delete with Filters

Use filters to delete only specific instances.
Always run with `dry_run=True` first to preview.

| Parameter | Description | Example |
|-----------|-------------|--------|
| `older_than_hours` | Only instances older than N hours | `2.0` |
| `owner` | Only instances created by this user | `"alice@example.com"` |
| `name_prefix` | Only instances whose name starts with prefix | `"test-"` |
| `status_filter` | Only this status (or set of statuses) | `"running"` |
| `dry_run` | Preview without deleting | `True` |

In [ ]:
# ── Step 4 │ Bulk Delete with Filters ──────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
# ----- Configure your filters here -----
OLDER_THAN_HOURS = 2       # delete instances older than 2 hours  (None = no age filter)
OWNER            = None    # e.g. "alice@example.com"             (None = all owners)
NAME_PREFIX      = None    # e.g. "test-"                         (None = all names)
STATUS_FILTER    = None    # e.g. "running"                       (None = all active)
DRY_RUN          = True    # set False to actually delete
# ----------------------------------------

client.instances.bulk_delete(
    older_than_hours=OLDER_THAN_HOURS,
    owner=OWNER,
    name_prefix=NAME_PREFIX,
    status_filter=STATUS_FILTER,
    dry_run=DRY_RUN,
)

## 4 — Confirm Final State

In [ ]:
# ── Step 5 │ Confirm Final State ───────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
remaining = [i for i in client.instances.list() if i["status"] in active_statuses]
if remaining:
    print(f"{len(remaining)} instance(s) still active:")
    for i in remaining:
        print(f"  id={i['id']}  status={i['status']}  owner={i.get('created_by','-')}")
else:
    print("No active instances — all wrapper pods terminated.")